<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/notebooks/01_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quiver Signals — Module 1: Foundations

**A signal-research system built on alternative data.**

This is the first notebook in a series that builds a complete quantitative trading
research pipeline from scratch. By the end of the series you will have a system that:

1. Ingests alternative data (congressional trades, insider transactions, government contracts, lobbying) from the Quiver Quantitative API
2. Converts that raw data into normalized **signal scores** on a $[0, 100]$ scale
3. Backtests those signals against historical prices to measure whether they carry real predictive edge
4. Classifies each candidate into a recommended trade structure (buy stock, sell puts, debit spread)
5. Surfaces everything in an interactive dashboard

Every notebook pairs plain-English explanation with the code that implements it, so the
repository reads as a narrative you can walk an interviewer through line by line.

---

### What this specific notebook does

Module 1 is pure plumbing — no finance yet. We set up the three pieces of
infrastructure the rest of the project depends on: a **GitHub repository** (the permanent
home for our code), a **Colab-to-GitHub workflow** (so we can write code in the browser
and version it properly), and a **clean project structure** (so the code stays organized
as it grows). Getting this right once saves friction in every session that follows.


## Why alternative data?

Traditional quantitative signals — price momentum, value ratios, earnings surprises — are
extracted from data that every fund on the planet already has. Edge there is thin because
the information is universally available the moment it is published.

**Alternative data** refers to non-traditional sources that correlate with future returns
before the broader market fully prices them in. A few examples we will use:

- **Congressional trading.** U.S. lawmakers file disclosures of their personal stock trades. Some committees sit close to information flow (defense appropriations, healthcare policy), and clustered buying across multiple members has historically preceded moves.
- **Insider transactions.** Corporate executives filing Form 4 buys with their own money signal conviction, especially when the buyer is a CEO or CFO rather than a junior officer.
- **Government contracts.** A freshly awarded federal contract is forward-looking revenue that may not yet be reflected in analyst estimates.
- **Lobbying spend.** Sharp increases in lobbying outlay often foreshadow regulatory catalysts.

The thesis of this project: each source is individually noisy, yet when several *independent*
sources point at the same ticker simultaneously, the combined signal becomes meaningful. We
formalize that intuition with a weighted **composite score** in Module 4.


## Before you run this notebook

Three one-time setup tasks happen outside Colab. Complete them first.

### 1. Create the GitHub repository
On **github.com**, create a new repository named `quiver-signals`. Mark it **Private**,
check **Add a README**, and create it. This is where all our work lives permanently.

### 2. Create a Personal Access Token (PAT)
Colab needs permission to push code to your repo. On GitHub go to
**Settings → Developer settings → Personal access tokens → Fine-grained tokens →
Generate new token**. Give it repository access to `quiver-signals` with
**Contents: Read and write** permission. Copy the token string (it appears once).

### 3. Store the token in Colab Secrets
In Colab, click the **🔑 key icon** in the left sidebar. Add a new secret named
`GITHUB_TOKEN`, paste your PAT as the value, and toggle **Notebook access** on.

> **Why a secret instead of pasting the token in a cell?** A token typed into a code cell
> gets saved into the notebook file and pushed to GitHub, exposing it publicly. The Secrets
> panel keeps it out of the notebook entirely. Treating credentials this carefully is exactly
> the habit an interviewer looks for.


## Step 1 — Configure Git

Git records *who* made each change. We set a name and email once per session. These do not
need to match your GitHub login exactly, though using the same email keeps your commit
history attributed to your account on GitHub's contribution graph.


In [ ]:
# Edit these two lines, then run the cell
!git config --global user.name "balla-a12"
!git config --global user.email "aballa1234@gmail.com"
print("Git identity configured")

Git identity configured


## Step 2 — Clone your repository securely

Cloning copies your GitHub repo into this Colab session so we can add files to it.

A subtle point on security: we read the token from Colab Secrets and build the clone URL
inside Python, then run the clone through `subprocess` so the token is **never printed** to
the notebook output. Compare that to `!git clone https://TOKEN@github.com/...`, where the
expanded token can leak into the visible cell output and then into version control.

One more thing worth understanding: the Colab filesystem is **ephemeral**. When this session
ends, everything here disappears. That is fine because GitHub is our permanent source of
truth — we push our work there before closing. Each new session begins by cloning fresh.


In [2]:
import subprocess, os
from google.colab import userdata

GITHUB_USER = "balla-a12"   # <-- edit this
REPO        = "quant-equity-research"

token = userdata.get("GITHUB_TOKEN")          # pulled from the Secrets panel
clone_url = f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"

result = subprocess.run(["git", "clone", clone_url],
                        capture_output=True, text=True)

if result.returncode == 0 or "already exists" in result.stderr:
    os.chdir(REPO)
    print("Cloned and entered:", os.getcwd())
else:
    print("Problem cloning:\n", result.stderr)

Cloned and entered: /content/quant-equity-research


## Step 3 — Build the project structure

A thoughtful directory layout signals engineering maturity. Here is what we create and the
reasoning behind each piece:

```
quiver-signals/
├── README.md            <- the front door; what an interviewer reads first
├── requirements.txt     <- exact packages, so the repo is reproducible
├── .gitignore           <- keeps secrets, data, and caches out of version control
├── notebooks/           <- the narrative layer: markdown + code, one per module
├── src/                 <- reusable Python modules the notebooks import
│   ├── ingestion/       <- code that pulls and normalizes Quiver data
│   ├── signals/         <- code that turns data into 0-100 scores
│   ├── backtest/        <- the historical simulation engine
│   └── utils/           <- shared helpers
└── data/                <- local database and cached prices (never committed)
```

The split between `notebooks/` and `src/` matters. Notebooks are wonderful for *explaining*
and *exploring*, yet they are awkward to reuse and test. So the durable logic lives in
`src/` as importable modules, and the notebooks import that logic and narrate it. This is
the pattern professional data teams use, and showing you understand it is a quiet credibility
signal.


In [3]:
import os

for d in ["notebooks", "src/ingestion", "src/signals", "src/backtest", "src/utils", "data"]:
    os.makedirs(d, exist_ok=True)

# An __init__.py marks a folder as an importable Python package
for pkg in ["src", "src/ingestion", "src/signals", "src/backtest", "src/utils"]:
    open(os.path.join(pkg, "__init__.py"), "a").close()

# .gitkeep lets us commit the empty data/ folder (Git ignores empty directories otherwise)
open("data/.gitkeep", "a").close()

print("Project structure created:")
for root, dirs, files in os.walk("."):
    if "/.git" in root: continue
    depth = root.count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")

Project structure created:
./
  notebooks/
  data/
  src/
    signals/
    utils/
    backtest/
    ingestion/


## Step 4 — Write the README

The README is the single most-read file in any repository. For a portfolio project it does
double duty as a pitch: it tells a hiring manager what you built, why it is interesting, and
how to run it, all before they read a line of code. We write a strong one now and refine it
as the project grows.

The `%%writefile` magic at the top of the next cell writes everything below it to a file
rather than executing it as Python.


In [4]:
%%writefile README.md
# Quiver Signals

A quantitative signal-research system built on alternative data. The pipeline ingests
congressional trades, insider transactions, government contracts, and lobbying activity
from the [Quiver Quantitative API](https://api.quiverquant.com), converts them into
normalized signal scores, backtests those signals against historical prices, and surfaces
ranked trade candidates in an interactive dashboard.

## Why this exists

Most quant signals are extracted from data every fund already owns, so the edge is thin.
This project instead mines *alternative* data sources that can correlate with future returns
before the broader market prices them in, and combines several independent sources into a
single weighted conviction score.

## Architecture

| Layer | Module | What it does |
|---|---|---|
| Ingestion | `src/ingestion` | Pull and normalize Quiver datasets |
| Signals | `src/signals` | Convert raw data into 0-100 scores |
| Backtest | `src/backtest` | Measure historical predictive edge |
| Dashboard | dashboard app | Visualize trending signals and candidates |

## Running it

The project ships with a **mock-data mode** so the full pipeline runs with no API key:

```bash
pip install -r requirements.txt
```

Open the notebooks in `notebooks/` in order, starting with `01_foundations.ipynb`.
To switch from mock data to live data, add a Quiver API token (see Module 2).

## Status

Built as a learning project, in public, one module at a time. See `notebooks/` for the
full narrative.

## Disclaimer

For research and educational purposes only. Nothing here is financial advice.


Overwriting README.md


## Step 5 — Pin dependencies and protect secrets

`requirements.txt` lists the exact packages the project needs, so anyone (including future
you on a fresh machine) can reproduce the environment with one command.

`.gitignore` lists files Git should **never** track. This is a critical security file: it
keeps API keys, the local database, and large cached data out of your public commit history.
A leaked key in a Git history is one of the most common and costly mistakes in the field, and
a clean `.gitignore` prevents it.


In [5]:
%%writefile requirements.txt
quiverquant
pandas
numpy
sqlalchemy
yfinance
plotly
scipy
python-dotenv
loguru


Writing requirements.txt


In [6]:
%%writefile .gitignore
# Secrets
.env
*.key

# Local database and cached data
*.db
data/*.parquet
data/*.csv
price_cache/

# Python
__pycache__/
*.pyc

# Notebook checkpoints
.ipynb_checkpoints/


Writing .gitignore


## Step 6 — Commit and push

Three commands move our work from this session up to GitHub:

- `git add -A` stages every change (new files, edits, deletions) for the next commit.
- `git commit -m "..."` bundles the staged changes into one labeled snapshot in history.
- `git push` uploads that snapshot to GitHub, where it lives permanently.

Think of a commit as a save point with a description. Good commit messages turn your Git
history into a readable changelog of how the project evolved, which is itself something an
interviewer may scroll through.


In [7]:
!git add -A
!git commit -m "Module 1: project foundations, structure, and README"
!git push

[main 128403b] Module 1: project foundations, structure, and README
 9 files changed, 68 insertions(+), 1 deletion(-)
 create mode 100644 .gitignore
 rewrite README.md (100%)
 create mode 100644 data/.gitkeep
 create mode 100644 requirements.txt
 create mode 100644 src/__init__.py
 create mode 100644 src/backtest/__init__.py
 create mode 100644 src/ingestion/__init__.py
 create mode 100644 src/signals/__init__.py
 create mode 100644 src/utils/__init__.py
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (9/9), 1.65 KiB | 1.65 MiB/s, done.
Total 9 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/balla-a12/quant-equity-research.git
   4505faf..128403b  main -> main


## Step 7 — Save this notebook into the repository

The git commands above pushed our files, yet this notebook is still living only in your Colab
session. To file it inside the repo's `notebooks/` folder, use Colab's built-in integration:

**File → Save a copy in GitHub →** choose the `quiver-signals` repository, set the path to
`notebooks/01_foundations.ipynb`, and confirm.

From now on, that menu item saves your latest notebook changes straight to GitHub. The git
CLI we used above is for everything that is *not* a notebook (the `src/` modules, README,
config files).


## Recap and what comes next

You now have a private GitHub repository with a professional structure, a portfolio-grade
README, pinned dependencies, a security-conscious `.gitignore`, and a clean Colab-to-GitHub
workflow. Everything from here lands in this foundation.

**Module 2 — The data layer.** We build the wrapper around the Quiver API and design the
database that stores what we pull. Crucially, we build a **mock-data mode** that generates
synthetic data shaped exactly like Quiver's real responses, so the entire pipeline runs with
no API key and no cost. That is what lets you keep building now and only purchase the Quiver
subscription when we are ready for the first live pull in Module 3.

Before moving on, confirm your repo on github.com shows the new folders, the README, and the
two config files. Once it does, you are ready for Module 2.
